### Data Engineering Pipeline – Data Quality

This notebook implements the data engineering pipeline for the data quality assessment.
The pipeline covers data ingestion, normalization of nested structures, detection and quantification of data quality issues, and remediation steps.
All steps are deterministic and can be re-run end-to-end.

### Data Quality Issues Identified and Remediated

The following issues were identified and addressed: missing values, duplicate records, inconsistent data types, non-atomic nested fields, and logical inconsistencies in decision-related attributes.
Remediation steps were applied based on the nature of each issue and are documented inline.

## Overview (Inhaltsverzeichnis)

1. Data Loading and Initial Inspection  
2. Duplicate Records  
3. Missing Values (Completeness)   
4. Nested Field Normalization (spending_behavior)  
5. Logical Consistency Checks (Decision Fields)  
6. Summary of Remediation

## 1. Data Loading and Initial Inspection

This section loads the raw JSON file, flattens it into a tabular format, and performs basic inspection to understand schema and potential data quality issues.

In [1]:
import pandas as pd
import json

with open("../data/raw_credit_applications.json", "r") as f:
    data = json.load(f)

df = pd.json_normalize(data)
df.head()

,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,...,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,...,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,...,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,...,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN
3,app_024,"[{'category': 'Fitness', 'amount': 575}]",NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,...,70,0.35,0,True,NaN,NaN,4.3,34000.0,NaN,NaN
4,app_184,"[{'category': 'Entertainment', 'amount': 463}]",2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,M,1999-05-21,10080,...,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN


In [2]:
df.info()
df.describe(include="all").T.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 502 entries, 0 to 501
Data columns (total 21 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   _id                               502 non-null    object 
 1   spending_behavior                 502 non-null    object 
 2   processing_timestamp              62 non-null     object 
 3   applicant_info.full_name          502 non-null    object 
 4   applicant_info.email              502 non-null    object 
 5   applicant_info.ssn                497 non-null    object 
 6   applicant_info.ip_address         497 non-null    object 
 7   applicant_info.gender             501 non-null    object 
 8   applicant_info.date_of_birth      501 non-null    object 
 9   applicant_info.zip_code           501 non-null    object 
 10  financials.annual_income          497 non-null    object 
 11  financials.credit_history_months  502 non-null    int64  
 12  financia

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
_id,502,500,app_042,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
spending_behavior,502,496,"[{'category': 'Utilities', 'amount': 786}]",2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
processing_timestamp,62,4,2024-01-15T00:00:00Z,59,NaN,NaN,NaN,NaN,NaN,NaN,NaN
applicant_info.full_name,502,475,Susan Flores,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
applicant_info.email,502,494,,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
applicant_info.ssn,497,494,937-72-8731,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
applicant_info.ip_address,497,496,192.168.91.142,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
applicant_info.gender,501,5,Male,195,NaN,NaN,NaN,NaN,NaN,NaN,NaN
applicant_info.date_of_birth,501,494,,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
applicant_info.zip_code,501,196,10048,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Initial data inspection** shows a total of 502 application records and 21 attributes.  
Several fields exhibit substantial missingness (e.g., processing_timestamp, loan_purpose, notes), while others are fully populated.  
Data types are heterogeneous, with multiple numeric attributes stored as object types, indicating potential consistency issues to be addressed in subsequent steps.

In [3]:
# Normalize column names by replacing dots with underscores for easier downstream handling
df.columns = df.columns.str.replace(".", "_", regex=False)

In [4]:
df.dtypes

_id                                  object
spending_behavior                    object
processing_timestamp                 object
applicant_info_full_name             object
applicant_info_email                 object
applicant_info_ssn                   object
applicant_info_ip_address            object
applicant_info_gender                object
applicant_info_date_of_birth         object
applicant_info_zip_code              object
financials_annual_income             object
financials_credit_history_months      int64
financials_debt_to_income           float64
financials_savings_balance            int64
decision_loan_approved                 bool
decision_rejection_reason            object
loan_purpose                         object
decision_interest_rate              float64
decision_approved_amount            float64
financials_annual_salary            float64
notes                                object
dtype: object

Data types were reviewed as part of the initial inspection.  
All attributes were found to be stored in formats consistent with their semantic meaning, and no type-related inconsistencies requiring remediation were identified.

## 2. Duplicate Records

In [5]:
# Create a working copy for data quality remediation
apps = df.copy()

# Identify duplicate application IDs
duplicate_ids = apps["_id"].duplicated(keep=False)

# Count duplicated records
duplicate_ids.sum()

np.int64(4)

In [6]:
apps.loc[duplicate_ids, "_id"].value_counts()

_id
app_042    2
app_001    2
Name: count, dtype: int64

In [7]:
# Remove duplicate records, keeping the first occurrence
apps = apps.drop_duplicates(subset="_id")

apps.shape

(500, 21)

Duplicate application records were assessed using the application identifier.  
Hard duplicates were quantified and removed to ensure record uniqueness prior to further data quality remediation.

## 3. Handling Missing Values (Completeness)

Missing values were quantified across all attributes to assess data completeness.  
Fields with extensive missingness were identified and remediation strategies were selected based on their relevance and usage.

In [8]:
missing = df.isna().sum().sort_values(ascending=False)
missing.head(15)

notes                               500
financials_annual_salary            497
loan_purpose                        452
processing_timestamp                440
decision_rejection_reason           292
decision_approved_amount            210
decision_interest_rate              210
financials_annual_income              5
applicant_info_ip_address             5
applicant_info_ssn                    5
applicant_info_date_of_birth          1
applicant_info_zip_code               1
applicant_info_gender                 1
spending_behavior                     0
financials_credit_history_months      0
dtype: int64

In [9]:
apps = apps.drop(columns=[
    "notes",
    "financials_annual_salary",
    "loan_purpose",
    "processing_timestamp"
])

apps.shape

(500, 17)

In [10]:
apps.isna().sum().sort_values(ascending=False).head(6)

decision_rejection_reason    292
decision_approved_amount     208
decision_interest_rate       208
financials_annual_income       5
applicant_info_ssn             4
applicant_info_ip_address      4
dtype: int64

Logical consistency between decision-related attributes was evaluated using a set of business rules.  
The following checks assess whether approval status, rejection reasons, and approval terms are mutually consistent.

In [11]:
bad_rejection_reason = (
    (apps["decision_loan_approved"] == False) &
    (apps["decision_rejection_reason"].isna())
).sum()

bad_approved_has_reason = (
    (apps["decision_loan_approved"] == True) &
    (apps["decision_rejection_reason"].notna())
).sum()

bad_approved_missing_terms = (
    (apps["decision_loan_approved"] == True) &
    (
        apps["decision_interest_rate"].isna() |
        apps["decision_approved_amount"].isna()
    )
).sum()

bad_rejection_reason, bad_approved_has_reason, bad_approved_missing_terms

(np.int64(0), np.int64(0), np.int64(0))

**Interpretation:**  
No logical inconsistencies were identified across the evaluated decision rules.  
Rejected applications consistently contain a rejection reason, while approved applications do not include rejection reasons and always provide the necessary approval terms.

In [12]:
apps["financials_annual_income"].isna().sum()
apps["financials_annual_income"].describe()

count       495
unique      132
top       79000
freq         11
Name: financials_annual_income, dtype: int64

In [13]:
# Impute missing annual income values using the median to reduce sensitivity to outliers
apps["financials_annual_income"] = apps["financials_annual_income"].fillna(
    apps["financials_annual_income"].median()
)

In [14]:
# Replace missing IP addresses with a placeholder to retain records while indicating missing information
apps["applicant_info_ip_address"] = apps["applicant_info_ip_address"].fillna("UNKNOWN")
apps["applicant_info_ip_address"].isna().sum()

np.int64(0)

In [15]:
apps.isna().sum().sort_values(ascending=False).head()

decision_rejection_reason    292
decision_approved_amount     208
decision_interest_rate       208
applicant_info_ssn             4
financials_annual_income       0
dtype: int64

**Handling of sensitive identifier attributes:**  
Missing values in the social security number (SSN) field were intentionally left unmodified.  
As an identifier rather than an analytical feature, SSNs cannot be meaningfully imputed, and retaining missing values avoids introducing invalid or misleading information.

## 4. Nested Field Normalization (spending_behavior)

The `spending_behavior` attribute contains nested lists of category–amount pairs, resulting in non-atomic values.  
To ensure a normalized, tabular structure, this field was expanded into a separate table with one record per spending category and application.

In [16]:
apps["spending_behavior"].iloc[0]

[{'category': 'Shopping', 'amount': 480},
 {'category': 'Rent', 'amount': 790},
 {'category': 'Alcohol', 'amount': 247}]

In [17]:
# Expand spending_behavior so each list element becomes a separate row
spend_exploded = apps[["_id", "spending_behavior"]].explode("spending_behavior")

In [18]:
# Convert nested dictionaries into separate columns
spend_flat = pd.concat(
    [
        spend_exploded["_id"].reset_index(drop=True),
        pd.json_normalize(spend_exploded["spending_behavior"])
    ],
    axis=1
)

In [19]:
print(spend_flat.head())
spend_flat.shape

       _id  category  amount
0  app_200  Shopping     480
1  app_200      Rent     790
2  app_200   Alcohol     247
3  app_037      Rent     608
4  app_037    Dining      96


(824, 3)

**Result:**  
The nested `spending_behavior` field was successfully normalized into a separate table with one record per spending category and application.  
The resulting table contains 824 records across 502 applications, enabling category-level analysis while preserving referential integrity via the application ID.

## 5. Data Quality Summary

This notebook implemented a reproducible data engineering pipeline to assess and remediate data quality issues in the credit application dataset.  
Key issues addressed include duplicate records, missing values, non-atomic nested fields, and logical inconsistencies in decision-related attributes. Remediation steps were applied conservatively on a working copy of the data to preserve the raw dataset, and all fixes were validated post-remediation.

The dataset contains personally identifiable information (PII), including names, email addresses, SSNs, IP addresses, dates of birth, and zip codes. These fields were identified during the data quality assessment but were not anonymized or transformed beyond necessary handling, as privacy and compliance remediation is addressed separately in the governance/privacy analysis.

## 6. Export Cleaned Datasets

The cleaned application-level dataset (`apps`) and the normalized spending behavior dataset (`spend_flat`) are exported to `data/processed` to enable downstream work by other team members without re-running the notebook.

In [20]:
import os

# Create output directory if it doesn't exist
os.makedirs("../data/processed", exist_ok=True)

# Export cleaned application-level data (1 row per application)
apps.to_csv("../data/processed/credit_applications_clean.csv", index=False)

# Export normalized spending behavior data (multiple rows per application, linked by _id)
spend_flat.to_csv("../data/processed/spending_behavior_normalized.csv", index=False)